In [1]:
%store -r

Unable to restore variable 'PD_DM_AF_BUILD_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'CBM_KG_CYPHER_DATA_FILE', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'COVID_DM_AF_BUILD_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'INTERFACE_INTERSECTION_UPSTREAM_AND_DOWNSTREAM_ANALYSIS_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'INTERFACE_GOAT_UPSTREAM_ANALYSIS_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'AD_KG_DATA_DIR', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'COVID_KG_CYPHER_DATA_FILE', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>
Unable to restore variable 'AD_KG_CYPHER_DATA_DIR', ignoring (use %store -d to forget!)
The error

In [2]:
import json
import os.path

import commute_dm.core
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

In [4]:
def write_interface_to_json_file(interface, output_file_path, description):
    """Serialize a `commute_dm.core.get_interface` result to a JSON file.

    `interface` is `{uniprot_id: [{"collection": <Collection node>,
    "entry": <CollectionEntry node>, "node": <Species node>,
    "subunit": <Protein node> or None}]}`. The join itself lives in
    `commute_dm.core.get_interface`; this notebook only archives it.

    `node` is the species standing for the protein, which is a complex whenever
    the collection holds the protein inside one; `subunit` is then the protein
    itself, and `None` otherwise. This is the only place `subunit` is read.
    """
    data = []
    for identifier, elements in interface.items():
        data.append(
            {
                "annotation": {"namespace": "uniprot", "identifier": identifier},
                "model_elements": [
                    {
                        "collection": element["collection"]["name"],
                        "entry_file_path": element["entry"]["file_path"],
                        "species_element_id": element["node"].element_id,
                        "species_name": element["node"]["name"],
                        "subunit_element_id": (
                            element["subunit"].element_id
                            if element["subunit"] is not None
                            else None
                        ),
                        "subunit_name": (
                            element["subunit"]["name"]
                            if element["subunit"] is not None
                            else None
                        ),
                    }
                    for element in elements
                ],
            }
        )
    output = {"description": description, "data": data}
    with open(output_file_path, "w") as f:
        json.dump(output, f)

We remake the directory where we store the interfaces:

In [5]:
commute_dm.utils.remake_dir(INTERFACE_DIR)

## Computing the interface between the maps based on annotations

### Interface between the COVID DM CD, the PD DM CD and the AD KG CD AF

We compute the interface between the COVID DM CD (or COVID DM CD AF), the PD DM CD (or COVID DM CD AF), and the AD KG CD AF, based on shared UniProt annotations, i.e., we query every UniProt id such that:
- the id is carried by a `:Protein` annotation key (`urn:miriam:uniprot:<id>`) in the `COVID_DM_CD` collection, and
- the same id is carried by a `:Protein` in the `PD_DM_CD` collection, and
- the same id is carried by a `:Protein` in the `AD_KG_CD_AF` collection.

For each such id we collect all the proteins (with their collection and entry) that carry it.

In [6]:
for collection_names, output_file_path in [
    (
        ["COVID_DM_CD", "PD_DM_CD", "AD_KG_CD_AF"],
        INTERFACE_DIR / "covid_pd_ad_af.json",
    ),
    (
        ["COVID_DM_CD_AF", "PD_DM_CD_AF", "AD_KG_CD_AF"],
        INTERFACE_DIR / "covid_af_pd_af_ad_af.json",
    ),
]:
    interface = commute_dm.core.get_interface(session, collection_names)
    print(collection_names, len(interface))
    write_interface_to_json_file(
        interface,
        output_file_path,
        (
            f"Interface between: {', '.join(collection_names)},\n"
            "based on shared UniProt annotations.\n"
            "This file is generated by the get_interfaces notebook.\n"
        ),
    )

['COVID_DM_CD', 'PD_DM_CD', 'AD_KG_CD_AF'] 111


['COVID_DM_CD_AF', 'PD_DM_CD_AF', 'AD_KG_CD_AF'] 111
